# Federated streaming detection: results analysis

FedAvg with per-client filters on top of a shared frozen scoring-model
snapshot.  The central question mirrors the streaming notebook: does each
client's filter respond to its assigned domain while keeping global
detection quality on par with no-filter?  Every cell degrades gracefully
when no federated runs have been produced yet.

Sections:

1. **Setup** -- run discovery, client partitioning, bootstrap vs stream composition.
2. **Summary tables** -- per-(variant, seed) metrics and cross-seed aggregate.
3. **Global mAP per round** -- detection performance convergence.
4. **Per-round mAP by filter family** -- grouped by manifest.
5. **Multi-seed mAP comparison** -- mean / min / max mAP across seeds.
6. **Per-class AP per round** -- Pedestrian AP as a domain-shift indicator.
7. **Per-client accept rates** -- how each client's filter responds to its domain.
8. **Per-client training effort** -- items processed, accepted, optimizer steps.
9. **Bandwidth efficiency** -- mAP vs total frames sent.
10. **Comparison with streaming (centralized)** -- federated vs centralized on matched manifests.
11. **Scoring-model refresh timeline** -- threshold evolution per round.

See `01_streaming_analysis.ipynb` for centralized-streaming results.

## 1 Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

sys.path.insert(0, str(Path.cwd() / "notebooks"))
if not (Path.cwd() / "pyproject.toml").exists():
    sys.path.insert(0, str(Path.cwd().parent / "notebooks"))

import analysis_helpers as ah

ah.setup_notebook_style()

PROJECT_ROOT = ah.find_project_root()
OUTPUTS = PROJECT_ROOT / "outputs"

SEED: int | None = None

# Variant -> color for per-variant plots.  Filter-family plots additionally
# use analysis_helpers.FILTER_FAMILY_COLORS to color by filter family
# (none / random / static / window / reservoir / uncertainty).
PALETTE: dict[str, str] = {
    # cityday_road_type manifest
    "fed_no_filter_cityday_road_type": "#2ca02c",
    "fed_random_filter_cityday_road_type": "#7f7f7f",
    "fed_dist_thresh_cityday_road_type_p10": "#1f77b4",
    "fed_adaptive_cityday_road_type_p10": "#f4a261",
    "fed_adaptive_cityday_road_type_p15": "#ff7f0e",
    "fed_adaptive_cityday_road_type_p20": "#8c564b",
    "fed_adaptive_cityday_road_type_p25": "#d62728",
    # citymix manifests
    "fed_no_filter_citymix_road_type": "#2ca02c",
    "fed_random_filter_citymix_road_type": "#7f7f7f",
    "fed_adaptive_citymix_road_type_p10": "#f4a261",
    "fed_adaptive_citymix_road_type_p25": "#d62728",
    "fed_no_filter_citymix_conditions": "#2ca02c",
    "fed_adaptive_citymix_conditions_p10": "#f4a261",
    "fed_adaptive_citymix_conditions_p25": "#d62728",
}


def _short_name(v: str) -> str:
    """Return a legend-friendly label derived from the variant name."""
    name = v[4:] if v.startswith("fed_") else v
    for manifest_tag, pretty in (
        ("cityday_curated", "city-day curated"),
        ("cityday_road_type", "city-day road_type"),
        ("citymix_road_type", "city-mix road_type"),
        ("citymix_conditions", "city-mix conditions"),
    ):
        if manifest_tag in name:
            core = name.replace(f"_{manifest_tag}", "")
            return f"{core} [{pretty}]"
    return name


SHORT_NAMES: dict[str, str] = {v: _short_name(v) for v in PALETTE}

print("Project:", PROJECT_ROOT)

In [ ]:
runs_df = ah.discover_runs(OUTPUTS)


def pick(pipeline: str, variant: str, seed: int | None = SEED) -> Path | None:
    return ah.pick_latest_run(runs_df, pipeline, variant, seed=seed)


def _runs_non_null(raw: dict[str, Path | None]) -> dict[str, Path]:
    out: dict[str, Path] = {}
    for k, v in raw.items():
        if v is not None:
            out[k] = v
    return out


# Variants in the primary RUN dict.  Non-existent variants are skipped
# automatically, so the list can include runs that have not yet been
# produced; only directories present under outputs/federated/ show up.
_VARIANTS: list[str] = [
    # cityday_road_type manifest
    "fed_no_filter_cityday_road_type",
    "fed_random_filter_cityday_road_type",
    "fed_adaptive_cityday_road_type_p10",
    "fed_adaptive_cityday_road_type_p15",
    "fed_adaptive_cityday_road_type_p20",
    "fed_adaptive_cityday_road_type_p25",
    # citymix manifests
    "fed_no_filter_citymix_road_type",
    "fed_random_filter_citymix_road_type",
    "fed_adaptive_citymix_road_type_p10",
    "fed_adaptive_citymix_road_type_p25",
    "fed_no_filter_citymix_conditions",
    "fed_adaptive_citymix_conditions_p10",
    "fed_adaptive_citymix_conditions_p25",
]
RUN: dict[str, Path] = _runs_non_null(
    {v: pick("federated", v) for v in _VARIANTS}
)

ROUNDS: dict[str, pd.DataFrame] = {}
for k, p in sorted(RUN.items()):
    rd = ah.read_csv(p / "rounds.csv")
    if rd is not None and not rd.empty:
        ROUNDS[k] = rd
        cfg = ah.load_run_config(p)
        print(f"{k:50s}  rounds={len(rd)}  seed={cfg.get('seed')}")
        print(f"{'':50s}  {p}")
    else:
        print(f"{k:50s}  [no rounds.csv]")

if not ROUNDS:
    print("No federated runs found under outputs/federated/ yet -- the "
          "remaining cells will render empty plots or skip cleanly.")
else:
    print(f"\nLoaded {len(ROUNDS)} federated runs.")

In [ ]:
# --- Seeds available per variant ---
# When a variant has multiple seeds, downstream plots can aggregate across
# seeds using ah.pick_runs_by_seed / ah.aggregate_across_seeds.  The RUN
# dict above holds the *latest* run per variant (at a single seed) and is
# what most cells in this notebook still use.
SEEDS_PER_VARIANT: dict[str, list[int]] = {
    v: ah.discover_seeds(runs_df, "federated", v) for v in RUN
}
for v, seeds in SEEDS_PER_VARIANT.items():
    tag = "multi-seed" if len(seeds) > 1 else "single"
    print(f"  {v:45s}  seeds={seeds}  [{tag}]")

MULTI_SEED = {v: seeds for v, seeds in SEEDS_PER_VARIANT.items() if len(seeds) >= 2}
print(f"\n{len(MULTI_SEED)} variants have >=2 seeds.")

In [ ]:
# --- Client partitioning summary ---
# Uses ah.client_ranges_from_config to reconstruct exactly the per-client
# stream ranges the experiment used (domain-aligned or contiguous).
# Each variant may reference a manifest with its own bootstrap size, so we
# resolve it per-run via ah.get_bootstrap_size.

for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if not man:
        continue
    boot_n = ah.get_bootstrap_size(man, cfg)
    ordering = man.get("ordering", {})
    block_order = ordering.get("block_order", [])
    block_sizes = ordering.get("block_sizes", {})
    n_train = sum(1 for f in man.get("frames", []) if f.get("split") == "train")
    n_stream = n_train - boot_n
    n_clients = int(cfg.get("num_clients", 4))

    ranges, groups, strategy = ah.client_ranges_from_config(man, cfg, boot_n)

    print(f"\n{k}  (bootstrap={boot_n})")
    print(f"  Strategy: {strategy} | Stream: {n_stream} frames over {n_clients} clients")
    print(f"  Manifest blocks: {block_order}")
    for cid, (s, e) in enumerate(ranges):
        g = groups[cid] if groups[cid] else ["(contiguous slice)"]
        sizes = ", ".join(f"{b}({block_sizes.get(b, e - s)})" for b in g)
        print(f"  Client {cid}: [{s}, {e}) size={e - s}  domains: {sizes}")

### Client domain composition

In [ ]:
from collections import Counter
from matplotlib.patches import Patch
import matplotlib.colors as mcolors

# Canonical palettes live in analysis_helpers so notebooks 00, 01, 02 stay
# color-consistent.  The federated client-composition plots additionally
# use ah.client_partitions to recover the exact per-client stream ranges
# from each run's config (contiguous or domain-aligned).
_DOMAIN_COLORS = ah.DOMAIN_COLORS
_TOD_COLORS = ah.TOD_COLORS


def _client_partitions(
    manifest: dict, cfg: dict,
) -> dict[int, pd.DataFrame]:
    """Return per-client stream-frame DataFrames using the run's strategy."""
    return ah.client_partitions(manifest, cfg)


def plot_client_domain_bars(
    manifest: dict,
    cfg: dict,
    title: str = "",
    field: str = "road_type",
    color_map: dict[str, str] | None = None,
) -> None:
    """Stacked horizontal bar: domain composition per client."""
    parts = _client_partitions(manifest, cfg)
    n_clients = len(parts)
    if color_map is None:
        color_map = _DOMAIN_COLORS if field == "road_type" else _TOD_COLORS
    all_cats: list[str] = []
    for cid in sorted(parts):
        for v in parts[cid][field].unique():
            if v not in all_cats:
                all_cats.append(v)

    fig, ax = plt.subplots(figsize=(10, 0.6 * n_clients + 1.2))
    for cid in sorted(parts):
        counts = Counter(parts[cid][field])
        total = sum(counts.values())
        left = 0.0
        for cat in all_cats:
            w = counts.get(cat, 0) / total
            ax.barh(cid, w, left=left, color=color_map.get(cat, "#aaaaaa"),
                    edgecolor="white", linewidth=0.3)
            if w > 0.08:
                ax.text(left + w / 2, cid, f"{w:.0%}", ha="center", va="center",
                        fontsize=7, color="white" if cat in ("night",) else "black")
            left += w
    ax.set_yticks(range(n_clients))
    ax.set_yticklabels([f"Client {i}" for i in range(n_clients)])
    ax.set_xlabel("Fraction of frames")
    ax.set_title(title or f"Client {field} composition")
    ax.legend(
        handles=[Patch(facecolor=color_map.get(c, "#aaa"), label=c) for c in all_cats],
        loc="upper right", fontsize=7, ncol=min(len(all_cats), 3),
    )
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def plot_client_stream_stripes(
    manifest: dict,
    cfg: dict,
    title: str = "",
    field: str = "road_type",
    color_map: dict[str, str] | None = None,
    downsample: int = 20,
) -> None:
    """Stripe plot: each row is a client, each pixel-column is a frame colored by domain.

    downsample controls the resolution (1 column per N frames).
    """
    parts = _client_partitions(manifest, cfg)
    n_clients = len(parts)
    if color_map is None:
        color_map = _DOMAIN_COLORS if field == "road_type" else _TOD_COLORS
    all_cats = list(color_map.keys())
    cat_to_idx = {c: i for i, c in enumerate(all_cats)}
    cmap = mcolors.ListedColormap([color_map.get(c, "#aaa") for c in all_cats])
    norm = mcolors.BoundaryNorm(range(len(all_cats) + 1), cmap.N)

    max_len = max(len(p) for p in parts.values())
    cols = (max_len + downsample - 1) // downsample
    img = np.full((n_clients, cols), np.nan)
    for cid in sorted(parts):
        vals = parts[cid][field].values
        for j in range(0, len(vals), downsample):
            chunk = vals[j : j + downsample]
            most_common = Counter(chunk).most_common(1)[0][0]
            img[cid, j // downsample] = cat_to_idx.get(most_common, np.nan)

    fig, ax = plt.subplots(figsize=(12, 0.8 * n_clients + 1.0))
    ax.imshow(img, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
    ax.set_yticks(range(n_clients))
    ax.set_yticklabels([f"Client {i}" for i in range(n_clients)])
    xticks_pos = np.linspace(0, cols - 1, 6).astype(int)
    ax.set_xticks(xticks_pos)
    ax.set_xticklabels([f"{int(x * downsample):,}" for x in xticks_pos])
    ax.set_xlabel("Frame index within client partition")
    ax.set_title(title or f"Client stream: {field}")
    ax.legend(
        handles=[Patch(facecolor=color_map.get(c, "#aaa"), label=c) for c in all_cats],
        loc="upper right", fontsize=7, ncol=min(len(all_cats), 3),
        bbox_to_anchor=(1.0, -0.08),
    )
    plt.tight_layout()
    plt.show()


def print_client_metadata_summary(manifest: dict, cfg: dict) -> None:
    """Print a compact summary of each client's metadata: ToD, weather, Pedestrian density."""
    parts = _client_partitions(manifest, cfg)
    rows = []
    for cid in sorted(parts):
        df = parts[cid]
        n = len(df)
        tod = Counter(df["time_of_day"])
        ped_frac = (df["num_pedestrians"] > 0).sum() / n if n else 0
        avg_ped = df["num_pedestrians"].mean()
        weather_top3 = Counter(df["scraped_weather"]).most_common(3)
        rows.append({
            "client": cid,
            "frames": n,
            "day%": f"{tod.get('day', 0)/n:.0%}",
            "dawn/dusk%": f"{tod.get('dawn/dusk', 0)/n:.0%}",
            "night%": f"{tod.get('night', 0)/n:.0%}",
            "ped_presence": f"{ped_frac:.0%}",
            "avg_ped": f"{avg_ped:.1f}",
            "top_weather": ", ".join(f"{w}({c/n:.0%})" for w, c in weather_top3),
        })
    display(pd.DataFrame(rows).set_index("client"))


# --- Generate for each loaded run ---
for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if not man:
        print(f"[skip] {k}: no manifest")
        continue
    n_clients = int(cfg.get("num_clients", 4))
    short = SHORT_NAMES.get(k, k)

    print(f"\n{'='*70}")
    print(f"  {short}  ({k})")
    print(f"{'='*70}")

    plot_client_domain_bars(man, cfg, title=f"{short} -- road type per client")
    plot_client_domain_bars(man, cfg, field="time_of_day", color_map=_TOD_COLORS,
                            title=f"{short} -- time of day per client")
    plot_client_stream_stripes(man, cfg, title=f"{short} -- stream domains (road type)")
    plot_client_stream_stripes(man, cfg, field="time_of_day", color_map=_TOD_COLORS,
                               title=f"{short} -- stream domains (time of day)")
    print_client_metadata_summary(man, cfg)
    print()

### Bootstrap vs. stream composition

In [ ]:
def plot_bootstrap_vs_stream(
    manifest: dict,
    bootstrap_frames: int = 5000,
    title: str = "",
) -> None:
    """Side-by-side bar charts comparing bootstrap and stream compositions."""
    boot, stream = ah.split_bootstrap_stream(manifest, bootstrap_frames)

    # Always show time_of_day.  Add the manifest's primary block field so
    # conditions manifests show their weather split too.
    primary = ah.primary_block_field(manifest)
    fields = [primary] + (["time_of_day"] if primary != "time_of_day" else [])
    if primary != "road_type":
        fields.append("road_type")
    fig, axes = plt.subplots(1, len(fields), figsize=(4 * len(fields), 3.5))
    if len(fields) == 1:
        axes = [axes]
    for ax, field in zip(axes, fields):
        cmap = ah.color_map_for_field(field)
        boot_counts = Counter(boot[field])
        stream_counts = Counter(stream[field])
        all_cats = sorted(set(list(boot_counts) + list(stream_counts)),
                          key=lambda c: -(boot_counts.get(c, 0) + stream_counts.get(c, 0)))
        x = np.arange(len(all_cats))
        n_boot = len(boot)
        n_stream = len(stream)
        boot_frac = [boot_counts.get(c, 0) / n_boot for c in all_cats]
        stream_frac = [stream_counts.get(c, 0) / n_stream for c in all_cats]

        w = 0.35
        bars_b = ax.bar(x - w / 2, boot_frac, w, label="Bootstrap",
                        color=[cmap.get(c, "#aaa") for c in all_cats],
                        edgecolor="black", linewidth=0.8, alpha=0.6)
        bars_s = ax.bar(x + w / 2, stream_frac, w, label="Stream",
                        color=[cmap.get(c, "#aaa") for c in all_cats],
                        edgecolor="black", linewidth=0.8, alpha=1.0)
        ax.set_xticks(x)
        ax.set_xticklabels(all_cats, rotation=30, ha="right", fontsize=7)
        ax.set_ylabel("Fraction")
        ax.set_title(field.replace("_", " ").title())
        ax.legend(fontsize=7)
    fig.suptitle(title or "Bootstrap vs. stream composition", fontsize=10, y=1.02)
    plt.tight_layout()
    plt.show()


# --- Show for each run (grouped by unique manifest) ---
_seen_manifests: set[str] = set()
for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    mpath = str(cfg.get("manifest_path", ""))
    if mpath in _seen_manifests:
        continue
    _seen_manifests.add(mpath)
    man = ah.load_manifest(PROJECT_ROOT, mpath)
    if not man:
        continue
    short = SHORT_NAMES.get(k, k)
    ordering = man.get("ordering", {})
    boot_n = ordering.get("bootstrap_frames", 5000)
    print(f"\nManifest: {mpath}")
    print(f"  Strategy: {ordering.get('strategy', '?')}")
    print(f"  Bootstrap: {boot_n} frames")
    plot_bootstrap_vs_stream(man, bootstrap_frames=boot_n,
                             title=f"Bootstrap vs. stream -- {mpath.split('/')[-1]}")

## 2 Summary tables

In [ ]:
# Per-(variant, seed) summary: filter family, manifest, accept rate,
# best / last / iso-compute mAP, refresh count, wall-clock duration.
# ah.variant_summary_table is the same function used by the streaming
# notebook and notebooks/analyze_runs.py.
# Per-manifest iso-compute budget: smallest final training-step count
# across filter runs of each manifest.
_filter_variants = [v for v in RUN.keys()
                    if ah.filter_mode(ah.load_run_config(ah.pick_latest_run(runs_df, 'federated', v)))
                    in {'static', 'window', 'reservoir'}]
_steps_by_manifest: dict[str, int] = {}
for _v in _filter_variants:
    for _rd in ah.pick_runs_by_seed(runs_df, 'federated', _v).values():
        _cfg = ah.load_run_config(_rd)
        _rd_df = ah.read_csv(_rd / 'rounds.csv')
        _s = ah.compute_step_series(_rd_df)
        if _s is None or _s.dropna().empty:
            continue
        _m = ah.manifest_family(_cfg)
        _last = int(_s.dropna().iloc[-1])
        _steps_by_manifest[_m] = min(_steps_by_manifest.get(_m, _last), _last)
per_seed = ah.variant_summary_table(
    runs_df, 'federated', list(RUN.keys()),
    target_optim_steps=_steps_by_manifest or None,
)
if per_seed.empty:
    print("No federated runs loaded.")
else:
    pretty = per_seed.drop(columns=["run_dir"]).copy()
    for c in ("accept_rate", "best_mAP", "last_mAP", "iso_mAP"):
        if c in pretty:
            pretty[c] = pretty[c].astype(float).round(4)
    print("Per-(variant, seed) summary:")
    print(pretty.to_string(index=False))

    agg = ah.aggregate_summary_across_seeds(per_seed)
    if not agg.empty:
        for c in agg.columns:
            if agg[c].dtype.kind == "f":
                agg[c] = agg[c].round(4)
        print("\nAggregated across seeds:")
        print(agg.to_string(index=False))

# Round-level totals (federated-only extras: client-level accepted/item
# counts aggregated from rounds.csv).
rows: list[dict] = []
for k, rd in sorted(ROUNDS.items()):
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    total_accepted = 0
    total_items = 0
    for cid in range(n_clients):
        acc_col = f"client_{cid}_accepted"
        items_col = f"client_{cid}_items"
        if acc_col in rd.columns:
            total_accepted += int(rd[acc_col].astype(float).sum())
        if items_col in rd.columns:
            total_items += int(rd[items_col].astype(float).sum())
    last_mAP50 = float(rd["mAP_50"].iloc[-1]) if "mAP_50" in rd.columns and rd["mAP_50"].notna().any() else float("nan")
    last_ped = float(rd["AP_Pedestrian"].iloc[-1]) if "AP_Pedestrian" in rd.columns and rd["AP_Pedestrian"].notna().any() else float("nan")
    rows.append({
        "variant": SHORT_NAMES.get(k, k),
        "rounds": len(rd),
        "clients": n_clients,
        "total_items": total_items,
        "total_accepted": total_accepted,
        "accept_rate": round(total_accepted / total_items, 4) if total_items else 0.0,
        "last_mAP_50": round(last_mAP50, 4),
        "last_AP_Pedestrian": round(last_ped, 4),
    })
if rows:
    print("\nRound-level totals (federated-only counters):")
    print(pd.DataFrame(rows).to_string(index=False))

## 3 Global mAP per round

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for k, rd in ROUNDS.items():
    c = PALETTE.get(k, "gray")
    label = SHORT_NAMES.get(k, k)
    if "mAP" in rd.columns:
        valid = rd[rd["mAP"].notna()]
        axes[0].plot(valid["round"].to_numpy(), valid["mAP"].astype(float).to_numpy(),
                     marker="o", ms=4, color=c, label=label)
    if "mAP_50" in rd.columns:
        valid = rd[rd["mAP_50"].notna()]
        axes[1].plot(valid["round"].to_numpy(), valid["mAP_50"].astype(float).to_numpy(),
                     marker="o", ms=4, color=c, label=label)

axes[0].set(xlabel="Round", ylabel="mAP", title="Global mAP per round")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set(xlabel="Round", ylabel="mAP@50", title="Global mAP@50 per round")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 4 Per-round mAP by filter family

In [ ]:
by_manifest: dict[str, list[str]] = {}
for v, rdir in RUN.items():
    cfg = ah.load_run_config(rdir)
    by_manifest.setdefault(ah.manifest_family(cfg), []).append(v)

if not by_manifest:
    print("No federated runs loaded.")
else:
    n_panels = len(by_manifest)
    fig, axes = plt.subplots(
        1, n_panels, figsize=(6.2 * n_panels, 4.2), squeeze=False,
    )
    for ax, (man, variants) in zip(axes[0], sorted(by_manifest.items())):
        for v in sorted(variants):
            rd = ROUNDS.get(v)
            if rd is None or rd.empty or "mAP" not in rd.columns:
                continue
            cfg = ah.load_run_config(RUN[v])
            fam = ah.filter_mode(cfg)
            color = PALETTE.get(v, ah.FILTER_FAMILY_COLORS.get(fam, "#444"))
            ax.plot(rd["round"], rd["mAP"], color=color, lw=1.3,
                    label=f"{SHORT_NAMES.get(v, v)} [{fam}]")
        ax.set_xlabel("round")
        ax.set_ylabel("global mAP")
        ax.set_title(man)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=6)
    plt.tight_layout()
    plt.show()

## 5 Multi-seed mAP comparison

In [ ]:
multi_seed_variants = [v for v in RUN if len(SEEDS_PER_VARIANT.get(v, [])) >= 2]

if not multi_seed_variants:
    print("No variants with >=2 seeds yet; re-run after the multi-seed campaign.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    for v in multi_seed_variants:
        runs_by_seed = ah.pick_runs_by_seed(runs_df, "federated", v)
        agg = ah.aggregate_across_seeds(runs_by_seed, "rounds.csv", "round", "mAP")
        if agg.empty:
            continue
        color = PALETTE.get(v)
        ax.plot(agg["round"], agg["mean"],
                label=f"{SHORT_NAMES.get(v, v)} (n={len(runs_by_seed)})",
                color=color, lw=1.4, marker="o", ms=3)
        ax.fill_between(agg["round"], agg["min"], agg["max"],
                        color=color, alpha=0.20, linewidth=0)
    ax.set_xlabel("round")
    ax.set_ylabel("mAP")
    ax.set_title("Federated mAP -- mean (line) and min/max (band) across seeds")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Per-seed peak mAP summary table
    rows: list[dict] = []
    for v in multi_seed_variants:
        runs_by_seed = ah.pick_runs_by_seed(runs_df, "federated", v)
        summ = ah.summary_across_seeds(runs_by_seed)
        if summ.empty or "best_mAP" not in summ.columns:
            continue
        mean = summ["best_mAP"].mean()
        std = summ["best_mAP"].std(ddof=0)
        rows.append({
            "variant": v,
            "n_seeds": len(summ),
            "best_mAP_mean": round(mean, 4),
            "best_mAP_std": round(std, 4),
            "best_mAP_seeds": [(int(s), round(m, 4)) for s, m in
                               zip(summ["seed"], summ["best_mAP"])],
        })
    if rows:
        print("\nPeak-mAP across seeds:")
        print(pd.DataFrame(rows).to_string(index=False))

## 6 Per-class AP per round

In [ ]:
TARGET_CLASSES = ["Vehicle", "Pedestrian", "VulnerableVehicle"]

fig, axes = plt.subplots(1, len(TARGET_CLASSES), figsize=(5 * len(TARGET_CLASSES), 4))
if len(TARGET_CLASSES) == 1:
    axes = [axes]

for i, cls_name in enumerate(TARGET_CLASSES):
    col = f"AP_{cls_name}"
    ax = axes[i]
    for k, rd in ROUNDS.items():
        if col not in rd.columns:
            continue
        valid = rd[rd[col].notna()]
        if valid.empty:
            continue
        c = PALETTE.get(k, "gray")
        label = SHORT_NAMES.get(k, k)
        ax.plot(valid["round"].to_numpy(), valid[col].astype(float).to_numpy(),
                marker="o", ms=4, color=c, label=label)
    ax.set(xlabel="Round", ylabel=f"AP_{cls_name}", title=cls_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## 7 Per-client accept rates

In [ ]:
_CLIENT_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
                  "#9467bd", "#8c564b", "#e377c2", "#7f7f7f"]


# Every filter run (static / window / reservoir / uncertainty); skip
# no_filter and random because their per-client accept rates are trivial.
dist_keys = [k for k in ROUNDS if ah.filter_mode(
    ah.load_run_config(RUN[k])) in {"static", "window", "reservoir", "uncertainty"}]

for k in dist_keys:
    rd = ROUNDS[k]
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if man:
        ordering = man.get("ordering", {})
        boot_n = ordering.get("bootstrap_frames", 5000)
        client_labels = ah.client_domain_labels(man, cfg, boot_n)
    else:
        client_labels = {i: f"Client {i}" for i in range(n_clients)}

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(SHORT_NAMES.get(k, k), fontsize=11)

    for cid in range(n_clients):
        items_col = f"client_{cid}_items"
        acc_col = f"client_{cid}_accepted"
        if items_col not in rd.columns or acc_col not in rd.columns:
            continue
        items = rd[items_col].astype(float)
        accepted = rd[acc_col].astype(float)
        rate = (accepted / items.replace(0, np.nan)).fillna(0)

        color = _CLIENT_COLORS[cid % len(_CLIENT_COLORS)]
        lbl = client_labels.get(cid, f"Client {cid}")
        axes[0].plot(rd["round"].to_numpy(), rate.to_numpy(),
                     marker="o", ms=3, color=color, label=lbl)
        axes[1].plot(rd["round"].to_numpy(), accepted.to_numpy(),
                     marker="o", ms=3, color=color, label=lbl)

    axes[0].set(xlabel="Round", ylabel="Accept rate", title="Per-client accept rate")
    axes[0].legend(fontsize=7)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_ylim(-0.05, 1.05)

    axes[1].set(xlabel="Round", ylabel="Frames accepted", title="Per-client accepted frames")
    axes[1].legend(fontsize=7)
    axes[1].grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

## 8 Per-client training effort

In [ ]:
for k in sorted(ROUNDS.keys()):
    rd = ROUNDS[k]
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if man:
        ordering = man.get("ordering", {})
        boot_n = ordering.get("bootstrap_frames", 5000)
        clabels = ah.client_domain_labels(man, cfg, boot_n)
    else:
        clabels = {i: f"Client {i}" for i in range(n_clients)}

    print(f"\n{SHORT_NAMES.get(k, k)}")
    effort_rows = []
    for cid in range(n_clients):
        items_col = f"client_{cid}_items"
        acc_col = f"client_{cid}_accepted"
        rej_col = f"client_{cid}_rejected"
        steps_col = f"client_{cid}_optimizer_steps"
        if items_col not in rd.columns:
            continue
        total_items = int(rd[items_col].astype(float).sum())
        total_acc = int(rd[acc_col].astype(float).sum()) if acc_col in rd.columns else 0
        total_rej = int(rd[rej_col].astype(float).sum()) if rej_col in rd.columns else 0
        total_steps = int(rd[steps_col].astype(float).sum()) if steps_col in rd.columns else 0
        rate = total_acc / total_items if total_items > 0 else 0
        effort_rows.append({
            "client": clabels.get(cid, f"Client {cid}"),
            "items": total_items,
            "accepted": total_acc,
            "accept_rate": f"{rate:.1%}",
            "rejected": total_rej,
            "opt_steps": total_steps,
        })
    display(pd.DataFrame(effort_rows).set_index("client"))

## 9 Bandwidth efficiency

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for k, rd in ROUNDS.items():
    cfg = ah.load_run_config(RUN[k])
    n_clients = int(cfg.get("num_clients", 4))
    c = PALETTE.get(k, "gray")
    label = SHORT_NAMES.get(k, k)

    # Cumulative total accepted across all clients
    cum_accepted = np.zeros(len(rd))
    for cid in range(n_clients):
        acc_col = f"client_{cid}_accepted"
        if acc_col in rd.columns:
            cum_accepted += rd[acc_col].astype(float).to_numpy()
    cum_accepted = np.cumsum(cum_accepted)

    if "mAP" in rd.columns:
        valid_mask = rd["mAP"].notna()
        ax.plot(cum_accepted[valid_mask.to_numpy()],
                rd.loc[valid_mask, "mAP"].astype(float).to_numpy(),
                marker="o", ms=4, color=c, label=label)

ax.set(xlabel="Cumulative frames accepted (all clients)",
       ylabel="Global mAP",
       title="Bandwidth efficiency: mAP vs total frames used")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 10 Comparison with streaming (centralized)

In [ ]:
# Load corresponding streaming runs for comparison.  Only variants on
# manifests that have federated counterparts are loaded here.
_FED_MANIFESTS = {
    ah.manifest_family(ah.load_run_config(p)) for p in RUN.values()
}
_STREAM_CANDIDATES = [
    # cityday_road_type
    "no_filter_cityday_road_type",
    "random_filter_cityday_road_type",
    "adaptive_cityday_road_type_p10",
    # citymix_road_type
    "no_filter_citymix_road_type",
    "random_filter_citymix_road_type",
    "adaptive_citymix_road_type_p10",
    # citymix_conditions
    "no_filter_citymix_conditions",
    "adaptive_citymix_conditions_p10",
]
STREAM_RUNS: dict[str, Path] = _runs_non_null({
    v: pick("streaming", v) for v in _STREAM_CANDIDATES
    if any(m in v for m in _FED_MANIFESTS) or not _FED_MANIFESTS
})

STREAM_CK: dict[str, pd.DataFrame] = {}
for k, p in STREAM_RUNS.items():
    ck = ah.read_csv(p / "checkpoints.csv")
    if ck is not None and not ck.empty:
        STREAM_CK[k] = ck
        print(f"Streaming: {k:45s}  {len(ck)} checkpoints")

# Compare final mAP
print("\n--- Final mAP comparison ---")
print(f"{'Experiment':<55s} {'mAP':>8s}  {'mAP@50':>8s}  {'AP_Ped':>8s}")
print("-" * 85)

for k, rd in sorted(ROUNDS.items()):
    last = rd[rd["mAP"].notna()].iloc[-1] if "mAP" in rd.columns and rd["mAP"].notna().any() else None
    if last is not None:
        ped = float(last.get("AP_Pedestrian", 0)) if "AP_Pedestrian" in rd.columns else 0
        print(f"[Fed]  {SHORT_NAMES.get(k, k):<50s} {float(last['mAP']):8.4f}  "
              f"{float(last['mAP_50']):8.4f}  {ped:8.4f}")

for k, ck in sorted(STREAM_CK.items()):
    if "mAP" in ck.columns and ck["mAP"].notna().any():
        last = ck[ck["mAP"].notna()].iloc[-1]
        ped = float(last.get("AP_Pedestrian", 0)) if "AP_Pedestrian" in ck.columns else 0
        print(f"[Str]  {k:<50s} {float(last['mAP']):8.4f}  "
              f"{float(last['mAP_50']):8.4f}  {ped:8.4f}")

## 11 Scoring-model refresh timeline

In [ ]:
adaptive_keys = [k for k in RUN if "adaptive" in k]

if not adaptive_keys:
    print("No adaptive federated runs loaded.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    x_col = "refresh_idx"
    for v in adaptive_keys:
        ref = ah.read_csv(RUN[v] / "refreshes.csv")
        if ref is None or ref.empty:
            continue
        color = PALETTE.get(v, "#333")
        label = SHORT_NAMES.get(v, v)
        xs: list[float] = [0.0]
        ys: list[float] = [float(ref["threshold_before"].iloc[0])]
        x_col = "round" if "round" in ref.columns else (
            "items_seen" if "items_seen" in ref.columns else "refresh_idx"
        )
        for _, r in ref.iterrows():
            xs += [float(r[x_col]), float(r[x_col])]
            ys += [float(r["threshold_before"]), float(r["threshold_after"])]
        axes[0].plot(xs, ys, color=color, lw=1.3, label=label)
        axes[1].plot(ref["refresh_idx"].to_numpy(),
                     ref["duration_seconds"].to_numpy(),
                     marker="o", ms=4, color=color, label=label)
    axes[0].set(xlabel=x_col, ylabel="Mahalanobis threshold",
                title="Scoring-model threshold evolution")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=7)
    axes[1].set(xlabel="refresh index", ylabel="duration (s)",
                title="Refresh latency")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
